In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rethinking_neural_id.paths import RepoPaths

paths = RepoPaths.default()
FIGURE_DIR = paths.results_root / "bias"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


from skdim.datasets import hyperSphere, hyperBall
from skdim.id import MLE, TwoNN

TRUE_DIMS = [2, 10, 20, 50, 100, 200, 300, 500, 750, 1000]

# Sample size per repetition (paper used n=1000)
N_SAMPLES = 5000

# Number of Monte Carlo repetitions per dimension
N_REPS = 30

# Defining the data manifold
manifold = "hyperball"

# Random seed for reproducibility
SEED = 42

# MLE neighborhood setting
MLE_N_NEIGHBORS = 20

# TwoNN: fraction of largest distances to discard (default ~0.1)
TWONN_DISCARD_FRAC = 0.1

rng = np.random.RandomState(SEED)

def estimate_once(X):
    """
    Run both estimators on one dataset X and return (mle_est, twonn_est).
    """
    # MLE
    mle = MLE()
    mle_est = mle.fit(X, n_neighbors=MLE_N_NEIGHBORS)

    # TwoNN
    twonn = TwoNN(discard_fraction=TWONN_DISCARD_FRAC)
    twonn_est = twonn.fit(X)

    return float(mle_est.dimension_), float(twonn_est.dimension_)

def simulate_and_estimate(m, n_samples, n_reps, base_seed):
    """
    For a single true dimension m: simulate n_reps datasets of size n_samples
    on the surface of the m-sphere and collect ID estimates.
    """
    mle_vals = np.empty(n_reps, dtype=float)
    twonn_vals = np.empty(n_reps, dtype=float)

    for r in range(n_reps):
        rs = base_seed + 1000*m + r
        if manifold == "hypersphere":
          X = hyperSphere(n=n_samples, d=m, random_state=rs)
        elif manifold == "hyperball":
          X = hyperBall(n=n_samples, d=m, random_state=rs)
        else:
          raise ValueError(f"Unknown manifold: {manifold}")

        mle_est, twonn_est = estimate_once(X)
        mle_vals[r] = mle_est
        twonn_vals[r] = twonn_est

    return mle_vals, twonn_vals

# run sims

results = {m: {} for m in TRUE_DIMS}

for m in TRUE_DIMS:
    print(f"Running m={m}")
    mle_vals, twonn_vals = simulate_and_estimate(
        m=m, n_samples=N_SAMPLES, n_reps=N_REPS, base_seed=SEED
    )
    results[m]["MLE"] = mle_vals
    results[m]["TwoNN"] = twonn_vals

In [ ]:
# summarize
def summarize(estimates_by_m):
    """
    Convert {m: array} -> arrays of means and stds aligned to TRUE_DIMS.
    """
    means = np.array([np.mean(estimates_by_m[m]) for m in TRUE_DIMS])
    sds   = np.array([np.std (estimates_by_m[m], ddof=1) for m in TRUE_DIMS])
    return means, sds

mle_means, mle_sds     = summarize({m: results[m]["MLE"] for m in TRUE_DIMS})
twonn_means, twonn_sds = summarize({m: results[m]["TwoNN"] for m in TRUE_DIMS})

true_dims = np.array(TRUE_DIMS)

# plot
plt.figure(figsize=(5.2, 3.2), dpi=400)

# MLE: solid blue with ±2 SD ribbons and mean markers
plt.plot(true_dims, mle_means, lw=1, label="MLE", ls="--", color="#1f77b4")
plt.fill_between(true_dims, mle_means - 2*mle_sds, mle_means + 2*mle_sds,
                 alpha=0.15, edgecolor="none", color="#1f77b4")

# TwoNN: red dash-dot with ±2 SD ribbons and diamond markers
plt.plot(true_dims, twonn_means, lw=1, label="TwoNN", ls="--", color="#d62728")
plt.fill_between(true_dims, twonn_means - 2*twonn_sds, twonn_means + 2*twonn_sds,
                 alpha=0.12, edgecolor="none", color="#d62728")
plt.plot(true_dims, mle_means, "o", ms=5, color="#1f77b4")
plt.plot(true_dims, twonn_means, "D", ms=5, mfc="white", mec="#d62728", mew=1.2, color="#d62728")

# Diagonal "ideal" line (estimated = true)
tmin, tmax = true_dims.min(), true_dims.max()
plt.plot([tmin, tmax], [tmin, tmax], "k--", lw=1, alpha=0.6)

plt.xlim(tmin - 0.5, tmax + 1)
plt.ylim(0, max(tmax + 2, 1.1 * np.max([mle_means + 2*mle_sds, twonn_means + 2*twonn_sds])))

plt.xlabel("True Intrinsic Dimension", fontsize=12)
plt.ylabel("Estimated Intrinsic Dimension", fontsize=12)
plt.legend(frameon=True)
plt.tight_layout()

figure_path = FIGURE_DIR / "id_est_bias_ball.png"
plt.savefig(figure_path)
plt.show()


## Different k-MLE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skdim.datasets import hyperSphere, hyperBall
from skdim.id import MLE, TwoNN

# configs
TRUE_DIMS = [2, 10, 20, 50, 100, 200, 300, 500, 750, 1000]

N_SAMPLES = 5000
N_REPS = 20

manifold = "hyperball"
SEED = 42

MLE_K_LIST = [5, 10, 20, 50]
TWONN_DISCARD_FRAC = 0.1

rng = np.random.RandomState(SEED)

# estimation
def estimate_once(X, k):
    """
    Run one MLE with neighbor parameter k and TwoNN on dataset X.
    """
    # MLE
    mle = MLE()
    mle_est = mle.fit(X, n_neighbors=k)

    # TwoNN
    twonn = TwoNN(discard_fraction=TWONN_DISCARD_FRAC)
    twonn_est = twonn.fit(X)

    return float(mle_est.dimension_), float(twonn_est.dimension_)

def simulate_and_estimate(m, n_samples, n_reps, base_seed):
    """
    For a single true dimension m: simulate n_reps datasets of size n_samples
    on the chosen manifold and collect ID estimates for all MLE ks and TwoNN.
    """
    mle_vals = {k: np.empty(n_reps, dtype=float) for k in MLE_K_LIST}
    twonn_vals = np.empty(n_reps, dtype=float)

    for r in range(n_reps):
        rs = base_seed + 1000*m + r
        if manifold == "hypersphere":
            X = hyperSphere(n=n_samples, d=m, random_state=rs)
        elif manifold == "hyperball":
            X = hyperBall(n=n_samples, d=m, random_state=rs)
        else:
            raise ValueError(f"Unknown manifold: {manifold}")

        for k in MLE_K_LIST:
            mle_est, twonn_est = estimate_once(X, k)
            mle_vals[k][r] = mle_est
        twonn_vals[r] = twonn_est

    return mle_vals, twonn_vals

# run sims
results = {m: {} for m in TRUE_DIMS}

for m in TRUE_DIMS:
    print(f"Running m={m}")
    mle_vals, twonn_vals = simulate_and_estimate(
        m=m, n_samples=N_SAMPLES, n_reps=N_REPS, base_seed=SEED
    )
    results[m]["MLE"] = mle_vals
    results[m]["TwoNN"] = twonn_vals

# summarize
def summarize(estimates_dict):
    """
    For each true dim m, and for each method (MLE per k or TwoNN),
    compute mean and std across repetitions.
    """
    summary = {}
    for m in TRUE_DIMS:
        mle_dict = results[m]["MLE"]
        for k in MLE_K_LIST:
            if f"MLE_k{k}" not in summary:
                summary[f"MLE_k{k}"] = ([], [])
            summary[f"MLE_k{k}"][0].append(np.mean(mle_dict[k]))
            summary[f"MLE_k{k}"][1].append(np.std(mle_dict[k], ddof=1))

        if "TwoNN" not in summary:
            summary["TwoNN"] = ([], [])
        summary["TwoNN"][0].append(np.mean(results[m]["TwoNN"]))
        summary["TwoNN"][1].append(np.std(results[m]["TwoNN"], ddof=1))

    # convert lists to arrays
    for key in summary:
        means, sds = summary[key]
        summary[key] = (np.array(means), np.array(sds))
    return summary

summary = summarize(results)

true_dims = np.array(TRUE_DIMS)


In [ ]:
# plot
plt.figure(figsize=(7.2, 3.2), dpi=600)

# blue palette for MLE curves
colors = {
    5:  "#08306b",
    10: "#2171b5",
    20: "#6baed6",
    50: "#c6dbef"
}

for k in MLE_K_LIST:
    mle_means, mle_sds = summary[f"MLE_k{k}"]
    plt.plot(true_dims, mle_means, lw=1, label=f"MLE (k={k})", color=colors[k])
    plt.fill_between(true_dims, mle_means - 2*mle_sds, mle_means + 2*mle_sds,
                     alpha=0.12, edgecolor="none", color=colors[k])
    plt.plot(true_dims, mle_means, "o", ms=3, color=colors[k])

# TwoNN in red
twonn_means, twonn_sds = summary["TwoNN"]
plt.plot(true_dims, twonn_means, lw=1, label="TwoNN", ls="--", color="#d62728")
plt.fill_between(true_dims, twonn_means - 2*twonn_sds, twonn_means + 2*twonn_sds,
                 alpha=0.12, edgecolor="none", color="#d62728")
plt.plot(true_dims, twonn_means, "D", ms=3, mfc="white", mec="#d62728", mew=1.2, color="#d62728")

# Diagonal ideal line
tmin, tmax = true_dims.min(), true_dims.max()
plt.plot([tmin, tmax], [tmin, tmax], "k--", lw=1, alpha=0.6)


plt.xlabel("True Intrinsic Dimension", fontsize=12)
plt.ylabel("Estimated Intrinsic Dimension", fontsize=12)

plt.xlim(tmin - 0.5, tmax + 1)
plt.ylim(0, max(tmax + 2, 1.1 * np.max([mle_means + 2*mle_sds, twonn_means + 2*twonn_sds])))

plt.legend(frameon=True, ncol=1, fontsize=9)
plt.tight_layout()

figure_path = FIGURE_DIR / "id_est_bias_ball_multiMLE.png"
plt.savefig(figure_path)
plt.show()

## Increasing Ambient Space

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from skdim.datasets import hyperSphere, hyperBall
from skdim.id import MLE, TwoNN

# config
TRUE_ID = 50
AMBIENT_DIMS = [100, 200, 300, 500, 750, 1000, 2000, 10000]

N_SAMPLES = 5000
N_REPS = 20
SEED = 42

manifold = "hyperball"    # "hypersphere" or "hyperball"

MLE_N_NEIGHBORS = 20
TWONN_DISCARD_FRAC = 0.1

rng = np.random.RandomState(SEED)

# helpers / estimators

def random_rotation(D, rs):
    """Random D×D orthonormal matrix via QR."""
    A = rs.normal(size=(D, D))
    Q, R = np.linalg.qr(A)
    # make Q a proper rotation (det=+1)
    signs = np.sign(np.diag(R))
    Q = Q * signs
    return Q

def embed_to_ambient(Xk, D, rs):
    """
    Pad X (n×k) with zeros to n×D and apply a random rotation in R^D.
    Keeps intrinsic dimension = k while changing ambient = D.
    """
    n, k = Xk.shape
    if D == k:
        return Xk
    Xpad = np.pad(Xk, ((0, 0), (0, D - k)))
    Q = random_rotation(D, rs)
    return Xpad @ Q

def estimate_once(X):
    """
    Run both estimators on one dataset X and return (mle_est, twonn_est).
    """
    # MLE
    mle = MLE()
    mle_est = mle.fit(X, n_neighbors=MLE_N_NEIGHBORS)

    # TwoNN
    twonn = TwoNN(discard_fraction=TWONN_DISCARD_FRAC)
    twonn_est = twonn.fit(X)

    return float(mle_est.dimension_), float(twonn_est.dimension_)

def simulate_and_estimate_k_in_D(k, D, n_samples, n_reps, base_seed):
    """Simulate on k-manifold, embed in R^D, estimate ID repeatedly."""
    mle_vals = np.empty(n_reps, dtype=float)
    tw_vals  = np.empty(n_reps, dtype=float)

    for r in range(n_reps):
        rs = np.random.RandomState(base_seed + 100000*D + r)
        # 1) simulate in R^k
        if manifold == "hypersphere":
            Xk = hyperSphere(n=n_samples, d=k, random_state=rs)
        elif manifold == "hyperball":
            Xk = hyperBall(n=n_samples, d=k, random_state=rs)
        else:
            raise ValueError(f"Unknown manifold: {manifold}")

        #  embed into R^D with a random rotation
        X = embed_to_ambient(Xk, D=D, rs=rs)

        # estimate id
        mle_vals[r], tw_vals[r] = estimate_once(X)

    return mle_vals, tw_vals

In [ ]:
# run sims

results = {D: {} for D in AMBIENT_DIMS}
for D in AMBIENT_DIMS:
    print(f"Ambient D={D}")
    mle_vals, tw_vals = simulate_and_estimate_k_in_D(
        k=TRUE_ID, D=D, n_samples=N_SAMPLES, n_reps=N_REPS, base_seed=SEED
    )
    results[D]["MLE"] = mle_vals
    results[D]["TwoNN"] = tw_vals

In [ ]:
# summarize results

def summarize(metric):
    means = np.array([results[D][metric].mean() for D in AMBIENT_DIMS])
    sds   = np.array([results[D][metric].std(ddof=1) for D in AMBIENT_DIMS])
    return means, sds

ambient = np.array(AMBIENT_DIMS)
mle_means, mle_sds     = summarize("MLE")
twonn_means, twonn_sds = summarize("TwoNN")

# plot
plt.figure(figsize=(7.2, 4.2), dpi=400)

# MLE curve + bands
plt.plot(ambient, mle_means, lw=1, label="MLE", ls="--", color="#1f77b4")
plt.fill_between(ambient, mle_means - 2*mle_sds, mle_means + 2*mle_sds,
                 alpha=0.15, edgecolor="none", color="#1f77b4")

# TwoNN curve + bands
plt.plot(ambient, twonn_means, lw=1, ls="--", label="TwoNN", color="#d62728")
plt.fill_between(ambient, twonn_means - 2*twonn_sds, twonn_means + 2*twonn_sds,
                 alpha=0.12, edgecolor="none", color="#d62728")

# Reference line at the true ID
plt.axhline(TRUE_ID, ls="--", lw=1.2, color="k", alpha=0.7)

plt.xscale("log")

tmin, tmax = ambient.min(), ambient.max()
plt.xlim(tmin - 0.5, tmax + 1)
plt.ylim(np.min(mle_means) - 5, TRUE_ID + 5)

plt.xlabel("Ambient space dimension (log scale)")
plt.ylabel("Estimated intrinsic dimension")
plt.title(f"Fixed true ID = {TRUE_ID}; manifold={manifold}; n={N_SAMPLES}; reps={N_REPS}")
plt.legend()
plt.tight_layout()

figure_path = FIGURE_DIR / "id_est_increase_ambient.png"
plt.savefig(figure_path)
plt.show()